# Codex SDK 程式碼分析

使用官方 openai-codex Python SDK。SDK 會啟動隨套件安裝的 Codex app-server。程式建立 thread，再執行任務。

學習目標：載入設定、建立代理、觀察結果，再測試 HTTP 服務。

依官方 Python 非同步範例使用 AsyncCodex。新增 LiteLLM 供應者設定。每次請求使用暫存 thread、唯讀沙箱與 deny_all 核准模式。

## 1. 準備環境

從本資料夾執行 `uv sync --locked --group notebook`。
再執行 `uv run --group notebook jupyter lab tutorial.ipynb`。

In [1]:
import sys
from pathlib import Path

assert sys.version_info[:2] == (3, 13), "請選擇 Python 3.13 核心。"
assert Path("agent.py").is_file(), "請從本專案資料夾啟動 Notebook。"
RUN_LIVE = False
print("Python 版本：", sys.version.split()[0])

Python 版本： 3.13.15


## 2. 檢查設定

設定來自環境變數或 `.env`。此步驟不輸出金鑰。
離線學習不需要有效設定。真實推論需要模型別名。

In [2]:
from settings import ConfigurationError, Settings

try:
    settings = Settings.load(codex=True)
    print("模型：", settings.model)
    print("時間限制：", settings.timeout)
except ConfigurationError as exc:
    settings = None
    print(str(exc))

模型： gpt-5.4-mini
時間限制： 120.0


In [5]:
settings

Settings(endpoint='http://localhost:30148', model='gpt-5.4-mini', timeout=120.0)

## 3. 讀取範例程式

這個函式沒有處理空清單。先觀察程式，再請模型分析。

In [8]:
from agent import WORKSPACE

print(WORKSPACE)
print((WORKSPACE / "sample.py").read_text())

/home/ian/agent-framework-experiment/codex-sdk/workspace
"""教學用程式。練習找出空清單的邊界條件。"""


def average(values: list[float]) -> float:
    return sum(values) / len(values)



## 4. 檢查 SDK 介面

`AsyncCodex` 啟動本機 app-server。
`thread_start` 建立任務。`thread.run` 執行提示詞。
本範例使用唯讀沙箱與暫存 thread。LiteLLM 必須支援 Responses API。

In [9]:
from openai_codex import ApprovalMode, Sandbox

from agent import runtime_config

if settings is not None:
    config = runtime_config(settings)
    print("SDK 已設定。金鑰只透過環境變數傳入 runtime。")
print("沙箱：", Sandbox.read_only.value)
print("核准模式：", ApprovalMode.deny_all.value)

SDK 已設定。金鑰只透過環境變數傳入 runtime。
沙箱： read-only
核准模式： deny_all


## 5. 執行真實任務

確認 `.env` 的端點、金鑰與模型名稱。
將第一個程式區塊中的 `RUN_LIVE` 改為 `True`。
重新執行設定步驟，再執行下列區塊。這會使用模型額度。

In [ ]:
from agent import run_agent

result = None
RUN_LIVE = True
if RUN_LIVE:
    result = await run_agent(
        "分析這段 Python 程式：def average(values): return sum(values) / len(values)。指出空清單的問題，並提出修正建議。"
    )
    print(result["output"])
else:
    print("已略過真實推論。將 RUN_LIVE 改為 True 可執行。")

## 6. 觀察結果

檢查工具結果或代理輸出。每次執行都是獨立任務。

In [ ]:
if result is not None:
    print(result["framework"])
    print(result["output"])
else:
    print("尚未執行真實任務。")

## 7. 測試 Docker 服務

在終端機執行 `docker compose up --build -d --wait`。
開啟 `http://localhost:8003/docs`。
先呼叫 `/health`，再呼叫 `/invoke`。
完成後執行 `docker compose down`。

若 LiteLLM 只監聽 Linux 主機的 `127.0.0.1`，請改用下列命令：

```bash
docker compose -f compose.yaml -f docker/compose.host.yaml up --build -d --wait
```

停止時使用 `docker compose -f compose.yaml -f docker/compose.host.yaml down`。


In [ ]:
RUN_HTTP = False
if RUN_HTTP:
    import httpx

    async with httpx.AsyncClient(timeout=180) as client:
        response = await client.post(
            "http://localhost:8003/invoke",
            json={
                "prompt": "分析這段 Python 程式：def average(values): return sum(values) / len(values)。指出空清單的問題，並提出修正建議。"
            },
        )
        response.raise_for_status()
        print(response.json())

## 8. 延伸練習

請模型為 average 提出測試案例。比較空清單與一般清單的處理方式。

## 官方來源

- [Codex 官方 SDK Quickstart](https://learn.chatgpt.com/docs/codex-sdk)
- [Codex 官方供應者設定](https://learn.chatgpt.com/docs/config-file/config-advanced)
- [LiteLLM 官方 Responses API 文件](https://docs.litellm.ai/docs/response_api)